Question 1a: find out number of requests per category 

In [1]:
import pandas as pd
import geopandas as gpd

#read data (.csv velech chli irrefüehrend wöu s isch kes csv meh sondern es pandas (= datestruktur))
meldungen_csv = gpd.read_file("data/raw/stzh.zwn_meldungen_p.json")
#print("Meldungen CSV:", meldungen_csv.columns)

#check
#display(meldungen_csv.head())

#eine leere Spalte an meldungen_csv anfügen 
meldungen_csv["anzahl_meldungen"] = 0
#display(meldungen_csv.head())

#leere Spalte mit Werten füllen
for kategorie in meldungen_csv["service_code"].unique():  #dür aui kategorie düregoo
    anzahl_meldungen = meldungen_csv[meldungen_csv["service_code"] == kategorie].shape[0] #azahl zile i dere kategorie zöue

    meldungen_csv.loc[ #mäudige i neui Spalte schribe,, loc wöu select integer based on position (glaub????)
        meldungen_csv["service_code"] == kategorie, 
        "anzahl_meldungen"] = anzahl_meldungen
    
    print(f"Kategorie: {kategorie}, Anzahl Meldungen: {anzahl_meldungen}")
    
display(meldungen_csv.head())



Kategorie: Strasse/Trottoir/Platz, Anzahl Meldungen: 9870
Kategorie: Abfall/Sammelstelle, Anzahl Meldungen: 27339
Kategorie: Grünflächen/Spielplätze, Anzahl Meldungen: 7238
Kategorie: Beleuchtung/Uhren, Anzahl Meldungen: 5407
Kategorie: Graffiti, Anzahl Meldungen: 3759
Kategorie: Signalisation/Lichtsignal, Anzahl Meldungen: 10975
Kategorie: Brunnen/Hydranten, Anzahl Meldungen: 1289
Kategorie: VBZ/ÖV, Anzahl Meldungen: 1882
Kategorie: Allgemein, Anzahl Meldungen: 3969
Kategorie: Schädlinge, Anzahl Meldungen: 895


,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry,anzahl_meldungen
0,1,1,20130314151615,20130404072505,20130412075930,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (8.48423 47.37404),9870
1,2,2,20130314151757,20130326140505,20130412080022,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (8.50819 47.39512),9870
2,3,4,20130315091416,20130315095505,20130412080810,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,https://www.zueriwieneu.ch/photo/4.0.jpeg?bfbb...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Beim Trotto: Beim Trottoir sind einige Randste...,https://www.zueriwieneu.ch/report/4,POINT (8.55959 47.40826),9870
3,4,5,20130315091715,20130320100505,20130412080905,2681754,1250376,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Par,Auf dem Parkplatz beim Waidspital sind einige ...,https://www.zueriwieneu.ch/photo/5.0.jpeg?e309...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Par: Auf dem Parkplatz beim Waidspital...,https://www.zueriwieneu.ch/report/5,POINT (8.52163 47.39913),9870
4,5,6,20130315103653,20130422182505,20130423135033,2683094,1247762,Abfall/Sammelstelle,Abfall/Sammelstelle,fixed - council,16624,Arbeitskist,Arbeitskiste ist rund herum verschmiert,https://www.zueriwieneu.ch/photo/6.0.jpeg?8e65...,Web interface,Dieses Graffiti wird von uns in den kommenden ...,Arbeitskist: Arbeitskiste ist rund herum versc...,https://www.zueriwieneu.ch/report/6,POINT (8.53889 47.37545),27339


Question 1b: find out number of requests per category and Kreis/Quartier 
- improvements: listen sortieren nach anzahl meldungen

In [2]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd

#read data 
quartiere_json = gpd.read_file("data/raw/stzh.adm_statistische_quartiere_v.json")
quartiere_csv = gpd.read_file("data/raw/stzh.adm_statistische_quartiere_v.csv")
quartiere_ch = quartiere_json.to_crs(epsg = 2056)
#print("Quartiere JSON:", quartiere_json.columns)
#print("Quartiere CSV:", quartiere_ch.columns)

#merge quartiere json und csv um sowohl räumliche daten als auch attribute wie kname zu haben
quartiere_gdf = quartiere_json.merge(
    quartiere_csv, on = "objid", how = "left")
quartiere_gdf = quartiere_gdf.set_geometry("geometry_x") #aktive geometrie wieder definieren
quartiere_gdf = quartiere_gdf.to_crs(epsg = 2056)
#print("Quartiere GDF:", quartiere_gdf.columns)

meldungen_gdf = gpd.GeoDataFrame(
    meldungen_csv,
    geometry=gpd.points_from_xy(meldungen_csv["e"], meldungen_csv["n"]),
    crs=quartiere_ch.crs)
meldungen_ch = meldungen_gdf.to_crs(epsg = 2056)

#crs prüfen
print("CRS Quartiere", quartiere_gdf.crs)
print("CRS Meldungen", meldungen_ch.crs)

#spatial join
meldungen_quartier_join = gpd.sjoin(
    meldungen_ch, #left
    quartiere_gdf, #right
    how = "inner", predicate = "intersects")
print("JOIN:", meldungen_quartier_join.columns)


#meldungen pro quartier und kreis 
meldungen_pro_quartier = (meldungen_quartier_join.groupby("qname").size().reset_index(name = "anzahl_meldungen_quartier"))
display(meldungen_pro_quartier)

meldungen_pro_kreis = (meldungen_quartier_join.groupby("kname").size().reset_index(name = "anzahl_meldungen_kreis"))
display(meldungen_pro_kreis)

#neue spalten zu join hinzufügen
join_updated = meldungen_quartier_join.merge(
    meldungen_pro_kreis,
    on = "kname",
    how = "left")

join_updated = join_updated.merge(
    meldungen_pro_quartier,
    on = "qname",
    how = "left")

meldungen_quartier_join = join_updated[["objectid", "requested_datetime", "e", "n", "service_code", "geometry", "anzahl_meldungen", "index_right", "qname", "qnr", "kname", "knr", "geometry_y", "anzahl_meldungen_quartier", "anzahl_meldungen_kreis"]]
display(join_updated.head())
print("Join Updated:", join_updated.columns)

CRS Quartiere EPSG:2056
CRS Meldungen EPSG:2056
JOIN: Index(['objectid', 'service_request_id', 'requested_datetime',
       'agency_sent_datetime', 'updated_datetime', 'e', 'n', 'service_code',
       'service_name', 'status', 'userid', 'title', 'detail', 'media_url',
       'interface_used', 'service_notice', 'description', 'url', 'geometry',
       'anzahl_meldungen', 'index_right', 'objid', 'objectid_x', 'objectid_y',
       'qname', 'qnr', 'kname', 'knr', 'geometry_y'],
      dtype='str')


,qname,anzahl_meldungen_quartier
0,Affoltern,2419
1,Albisrieden,2048
2,Alt-Wiedikon,2502
3,Altstetten,4094
4,City,1741
5,Enge,2756
6,Escher Wyss,1562
7,Fluntern,1254
8,Friesenberg,1415
9,Gewerbeschule,2230


,kname,anzahl_meldungen_kreis
0,Kreis 1,5738
1,Kreis 10,6394
2,Kreis 11,8026
3,Kreis 12,2764
4,Kreis 2,6225
5,Kreis 3,9152
6,Kreis 4,10569
7,Kreis 5,3792
8,Kreis 6,5110
9,Kreis 7,5714


,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,...,objid,objectid_x,objectid_y,qname,qnr,kname,knr,geometry_y,anzahl_meldungen_kreis,anzahl_meldungen_quartier
0,1,1,20130314151615,20130404072505,20130412075930,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,23,16,16,Albisrieden,91,Kreis 9,9,"POLYGON ((2677463 1246898.9,2677484.8 1246940....",6142,2048
1,2,2,20130314151757,20130326140505,20130412080022,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,28,21,21,Höngg,101,Kreis 10,10,"POLYGON ((2677686.8 1251901,2677687.5 1251901....",6394,3090
2,3,4,20130315091416,20130315095505,20130412080810,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,8,33,33,Saatlen,121,Kreis 12,12,"POLYGON ((2684510.2 1251962.2,2684542.5 125196...",2764,676
3,4,5,20130315091715,20130320100505,20130412080905,2681754,1250376,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,29,22,22,Wipkingen,102,Kreis 10,10,"POLYGON ((2681180.2 1250369.6,2681181 1250370....",6394,3304
4,5,6,20130315103653,20130422182505,20130423135033,2683094,1247762,Abfall/Sammelstelle,Abfall/Sammelstelle,fixed - council,...,25,18,18,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",5738,1741


Join Updated: Index(['objectid', 'service_request_id', 'requested_datetime',
       'agency_sent_datetime', 'updated_datetime', 'e', 'n', 'service_code',
       'service_name', 'status', 'userid', 'title', 'detail', 'media_url',
       'interface_used', 'service_notice', 'description', 'url', 'geometry',
       'anzahl_meldungen', 'index_right', 'objid', 'objectid_x', 'objectid_y',
       'qname', 'qnr', 'kname', 'knr', 'geometry_y', 'anzahl_meldungen_kreis',
       'anzahl_meldungen_quartier'],
      dtype='str')


Question 2: requests in kreisen around the lake (difference between different lakesides)
"close to the lake" = Kreise 1,2,8 (chinawiese = kreis 8)

In [3]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

#read file
quartiere_gdf
quartiere_subset = quartiere_gdf[["objid", "geometry_x", "kname", "knr"]]


fläche_csv = pd.read_csv("data/raw/bevölkerung_zh2.csv")
print("Fläche:", fläche_csv.columns)
fläche_subset = fläche_csv[fläche_csv["RaumLang"].str.contains("Kreis", na = False)]
#display(fläche_subset)
#display(fläche_csv)

#Join Fläche und sonstige Geoinfos
join_fläche_meldungen = fläche_subset.merge(
    meldungen_quartier_join[["kname", "anzahl_meldungen_kreis"]],
    left_on = "RaumLang", 
    right_on = "kname", 
    how = "left")

#Meldungsdichte pro Hektar
join_fläche_meldungen["meldungsdichte_kreis"] = join_fläche_meldungen["anzahl_meldungen_kreis"]/join_fläche_meldungen["FlaecheT"]
display(join_fläche_meldungen) 
print("join Meldungsdichte:", join_fläche_meldungen.columns)

#Meldungsdichte mergen
quartiere_subset = quartiere_subset.merge(
    join_fläche_meldungen[["meldungsdichte_kreis", "RaumLang"]],
    left_on = "kname", 
    right_on = "RaumLang",
    how = "inner")


Fläche: Index(['RaumKategorie', 'RaumSort', 'RaumLang', 'StichtagDatJahr', 'FlaecheT',
       'FlaecheL', 'FlaecheS'],
      dtype='str')


,RaumKategorie,RaumSort,RaumLang,StichtagDatJahr,FlaecheT,FlaecheL,FlaecheS,kname,anzahl_meldungen_kreis,meldungsdichte_kreis
0,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
1,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
2,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
3,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
4,Stadtkreis,10,Kreis 1,2009,180.0,155.8,88.0,Kreis 1,5738,31.877778
...,...,...,...,...,...,...,...,...,...,...
1234586,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920
1234587,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920
1234588,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920
1234589,Stadtkreis,120,Kreis 12,2025,596.6,421.4,272.2,Kreis 12,2764,4.632920


join Meldungsdichte: Index(['RaumKategorie', 'RaumSort', 'RaumLang', 'StichtagDatJahr', 'FlaecheT',
       'FlaecheL', 'FlaecheS', 'kname', 'anzahl_meldungen_kreis',
       'meldungsdichte_kreis'],
      dtype='str')


Question 2: visualization 
GEIT NED, STÜRZT AB SOBAUD IS WETT PLOTTE

In [ ]:
#Choropleth Map
fig, ax = plt.subplots(figsize=(10, 8)) #set up figure and axes

dmin = quartiere_subset["meldungsdichte_kreis"].quantile(0.02)
dmax = quartiere_subset["meldungsdichte_kreis"].quantile(0.98)

legend_options = {
    "label": "Report Density per Stadtkreis Normalized per Population",
    "orientation": "horizontal",
    "shrink": 0.6,
    "pad" : 0.05}

charte_plot = quartiere_subset.plot(
    ax = ax, 
    column = "meldungsdichte_kreis",
    vmin = dmin,
    vmax = dmax,
    cmap = "viridis",
    legend = True,
    legend_kwds = legend_options,
    edgecolor = "grey",
    linewidth = 0.1
)



Question 3: is the report density around the lake higher in august (summer) than in january (winter)?
QUESTION: I HA D FUNKTION UND MACHES MIT ERE LÄÄRE LISTE. I WETT ABER KE LISTE SONDERN ES DATAFRAME. CHANI S PRINZIP ÜBERNÄÄ UND WENN JO, WIE MACHI S GLICHE EIFACH MIT EMNE DATAFRAME (STATT LISTE)

In [46]:
import geopandas as gpd
import pandas as pd


#meldungen quartier join
#print(meldungen_quartier_join.head())
meldungen_quartier_join["requested_datetime"] = pd.to_datetime(
    meldungen_quartier_join["requested_datetime"], format = "%Y%m%d%H%M%S")

#close to lake area
kreise_see = meldungen_quartier_join[meldungen_quartier_join["knr"].isin(["1", "2", "8"])]
display(kreise_see)

#check
meldung_jahr = meldungen_quartier_join["requested_datetime"].dt.year
maldung_datum = meldungen_quartier_join["requested_datetime"].dt.date

#August und Januar (by hand)
august = meldungen_quartier_join[(meldungen_quartier_join["requested_datetime"] >= "2023-08-01") & 
                                 (meldungen_quartier_join["requested_datetime"] < "2023-09-01")].copy
januar = meldungen_quartier_join[(meldungen_quartier_join["requested_datetime"] >= "2023-01-01") & 
                                 (meldungen_quartier_join["requested_datetime"] < "2023-02-01")].copy

#display(august.head())

#Autust und Januar(function)
#def filter_by_month(data, year = None, month = None):
#    result = []

#    for row in data:
#        if year is not None and row["requested_datetime"].year != year:
#            continue
#        if month is not None and row["requested_datetime"].month != month:
#            continue 
#        result.append(row)
        
#    return pd.DataFrame(result)

#august_daten = filter_by_month(meldungen_quartier_join, month = 8)
#januar_daten = filter_by_month(meldungen_quartier_join, month = 1)


,objectid,requested_datetime,e,n,service_code,geometry,anzahl_meldungen,index_right,qname,qnr,kname,knr,geometry_y,anzahl_meldungen_quartier,anzahl_meldungen_kreis
4,5,2013-03-15 10:36:53,2683094,1247762,Abfall/Sammelstelle,POINT (2683094 1247762),27339,15,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",1741,5738
5,6,2013-03-16 17:54:42,2683475,1247422,Strasse/Trottoir/Platz,POINT (2683475 1247422),9870,13,Rathaus,11,Kreis 1,1,"POLYGON ((2683316.2 1247632.6,2683319.8 124763...",1457,5738
6,7,2013-03-16 18:04:21,2683303,1247675,Strasse/Trottoir/Platz,POINT (2683303 1247675),9870,24,Lindenhof,13,Kreis 1,1,"POLYGON ((2683037 1247571.6,2683044.5 1247600,...",1125,5738
14,15,2013-03-21 09:06:05,2684556,1245435,Strasse/Trottoir/Platz,POINT (2684556 1245435),9870,9,Mühlebach,82,Kreis 8,8,"POLYGON ((2683784.2 1246610.5,2683801.5 124663...",908,2997
19,20,2013-03-27 10:34:48,2683538,1247650,Strasse/Trottoir/Platz,POINT (2683538 1247650),9870,13,Rathaus,11,Kreis 1,1,"POLYGON ((2683316.2 1247632.6,2683319.8 124763...",1457,5738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72612,72613,2026-05-01 07:48:09,2682625,1247058,Signalisation/Lichtsignal,POINT (2682625 1247058),10975,15,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",1741,5738
72613,72614,2026-05-01 08:39:59,2683133,1247858,Brunnen/Hydranten,POINT (2683133 1247858),1289,24,Lindenhof,13,Kreis 1,1,"POLYGON ((2683037 1247571.6,2683044.5 1247600,...",1125,5738
72615,72616,2026-05-01 16:23:20,2682876,1244477,Grünflächen/Spielplätze,POINT (2682876 1244477),7238,8,Wollishofen,21,Kreis 2,2,"POLYGON ((2681417 1244793.5,2681418 1244817.4,...",2837,6225
72619,72620,2026-05-01 23:33:55,2682506,1244158,Allgemein,POINT (2682506 1244158),3969,8,Wollishofen,21,Kreis 2,2,"POLYGON ((2681417 1244793.5,2681418 1244817.4,...",2837,6225


Question 3: visualization 

Question 4: Which trend does the summer data around the lake show?
CHANI ERST RICHTIG FERTIG MACHE WENN D FUNKTION VO QUESTION 3 FUNKTIONIERT

In [ ]:
#meldungen_quartier_join hat daten schon geparsed
import numpy as np
import pandas as pd

#check ob august ready ist fürs resampling
#display(august_daten.head())
#print(f"August: {august_daten.columns}")
print(f"August Index: {august_daten.index}") #DatetimeIndex (dtype = 'datetype64', name = 'requested_datetime')

#print(meldungen_quartier_join["knr"].dtype)
#kreise in seenähe auswählen
meldungen_quartier_join_see = meldungen_quartier_join[meldungen_quartier_join["kname"].isin(["Kreis 1","Kreis 2","Kreis 8"])]
#display(meldungen_quartier_join_see.head)

#Monatsmittelwerte resamplen
august_daten["august_mean"] = august_daten["anzahl_meldungen_kreis"].resample("ME").mean().dropna()
augusts_passed = np.arange(len(august_mean))
slope, intercept = np.polyfit(augusts_passed, august_mean, 1)
print(f"long-term request trend: {slope:.3f} per year")
print(f"total change of number of requests over the dataset: {(slope * len(augusts_passed)):.2f}")

August Index: DatetimeIndex(['2023-08-08 16:39:04', '2023-08-08 22:57:52',
               '2023-08-02 16:28:36', '2023-08-04 14:45:19',
               '2023-08-09 12:48:30', '2023-08-09 12:49:06',
               '2023-08-01 09:12:51', '2023-08-01 13:49:49',
               '2023-08-01 15:31:45', '2023-08-01 15:31:59',
               ...
               '2023-08-31 16:59:47', '2023-08-31 17:00:53',
               '2023-08-31 17:01:51', '2023-08-31 17:03:08',
               '2023-08-31 17:04:41', '2023-08-31 17:07:14',
               '2023-08-31 17:08:31', '2023-08-31 18:46:36',
               '2023-08-31 20:15:23', '2023-08-31 21:31:14'],
              dtype='datetime64[us]', name='requested_datetime', length=775, freq=None)


c:\Users\User\miniconda3\envs\sds210\Lib\site-packages\numpy\lib\_polynomial_impl.py:674: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


LinAlgError: SVD did not converge in Linear Least Squares